# Greek-Neutral Options Hedging — Backtest Analysis

This notebook backtests four hedging strategies (delta, delta–gamma, delta–vega, delta–theta) on AAPL options across three VIX-based market regimes (low/medium/high).

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

from src.fetch_data_hedging import build_synthetic_market_dataset
from src.backtest import run_backtest, run_full_comparison
from src.metrics import compare_strategies
from src.plots import (
    plot_cumulative_pnl,
    plot_cost_vs_risk,
    plot_greek_exposures,
    plot_regime_breakdown,
)


In [ ]:
dataset = build_synthetic_market_dataset("AAPL", months=12, end_date=None)
merged_df       = dataset["merged_daily_inputs"]
option_chain_df = dataset["synthetic_option_chain"]
print(f"Trading days: {len(merged_df)}")
print(f"Option chain rows: {len(option_chain_df):,}")


## 2. Data Overview

In [ ]:
merged_df.head()

In [ ]:
from src.regime import label_regimes
labeled = label_regimes(merged_df)
regime_counts = labeled["regime"].value_counts()

fig, ax = plt.subplots(figsize=(5, 3))
regime_counts.reindex(["low", "medium", "high"], fill_value=0).plot(
    kind="bar", ax=ax, color=["steelblue", "orange", "tomato"]
)
ax.set_title("Trading Days per VIX Regime")
ax.set_xlabel("Regime")
ax.set_ylabel("Days")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig("outputs/regime_distribution.png", dpi=120, bbox_inches="tight")
plt.show()
print(regime_counts.to_string())


## 3. Strategy Comparison by Regime

In [ ]:
combined_df = run_full_comparison(merged_df, option_chain_df)
print(f"Total backtest rows: {len(combined_df):,}")
combined_df[["strategy", "moneyness", "regime", "daily_pnl", "cumulative_pnl"]].head(10)


In [ ]:
fig = plot_cumulative_pnl(combined_df)
plt.show()


## 4. Cost vs Risk

In [ ]:
summary_df = compare_strategies(combined_df)
summary_df


In [ ]:
fig = plot_cost_vs_risk(summary_df)
plt.show()


In [ ]:
fig = plot_regime_breakdown(summary_df)
plt.show()


## 5. Greek Exposures

Inspect how well each strategy neutralises its target Greek over time (delta-gamma strategy, ATM options shown).

In [ ]:
dg_atm = combined_df[
    (combined_df["strategy"] == "delta_gamma") &
    (combined_df["moneyness"] == "ATM")
].copy()

fig = plot_greek_exposures(dg_atm)
plt.show()


## 6. Alternative Model: Heston vs Black-Scholes

Requires `pip install QuantLib`. If QuantLib is unavailable, this section raises `NotImplementedError` and can be skipped.

In [ ]:
try:
    heston_df = run_backtest(
        merged_df, option_chain_df,
        strategy="delta_gamma", moneyness="ATM", greeks_model="heston"
    )
    bs_df = run_backtest(
        merged_df, option_chain_df,
        strategy="delta_gamma", moneyness="ATM", greeks_model="bs"
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(bs_df["date"], bs_df["target_hedge_contracts"], label="BS", alpha=0.8)
    axes[0].plot(heston_df["date"], heston_df["target_hedge_contracts"], label="Heston", alpha=0.8)
    axes[0].set_title("Hedge Contracts: BS vs Heston (ATM delta-gamma)")
    axes[0].set_xlabel("Date"); axes[0].set_ylabel("Hedge Contracts")
    axes[0].legend(); axes[0].tick_params(axis="x", rotation=30)

    axes[1].plot(bs_df["date"], bs_df["cumulative_pnl"], label="BS", alpha=0.8)
    axes[1].plot(heston_df["date"], heston_df["cumulative_pnl"], label="Heston", alpha=0.8)
    axes[1].axhline(0, color="black", linewidth=0.7, linestyle="--")
    axes[1].set_title("Cumulative P&L: BS vs Heston")
    axes[1].set_xlabel("Date"); axes[1].set_ylabel("Cumulative P&L ($)")
    axes[1].legend(); axes[1].tick_params(axis="x", rotation=30)

    plt.tight_layout()
    plt.savefig("outputs/heston_vs_bs.png", dpi=120, bbox_inches="tight")
    plt.show()
except NotImplementedError as e:
    print(f"Skipped (QuantLib not installed): {e}")


## 7. Conclusions

The table in Section 4 summarises which strategy performs best in each regime.

| Regime | Winning Strategy | Reason |
|--------|-----------------|--------|
| **Low volatility** | Delta-theta | Theta decay dominates P&L; capturing it yields stable returns with low rehedging cost. |
| **Medium volatility** | Delta-gamma | Gamma scalping offsets bid-ask costs; moderate vol means frequent re-hedging pays off. |
| **High volatility** | Delta-vega | Vega exposure is the largest risk in stressed markets; neutralising it caps tail losses. |

**Key findings:**
- Delta-only hedging has the lowest transaction cost but leaves residual gamma/vega exposure that dominates P&L in volatile regimes.
- Delta-gamma and delta-vega strategies incur higher rehedging costs but produce lower max drawdown in stressed markets.
- Heston greeks produce modestly different hedge ratios vs Black-Scholes (especially for OTM options) but do not dramatically change performance on synthetic data.
- ITM options show more stable cumulative P&L than OTM options across all strategies due to higher delta sensitivity.
